In [1]:
import json
import numpy as np
from scipy.sparse import load_npz

train_matrix = load_npz("../data/processed/train_interactions.npz")

with open("../data/processed/user_idx_map.json") as f:
    user_idx_map = json.load(f)
with open("../data/processed/item_idx_map.json") as f:
    item_idx_map = json.load(f)
with open("../data/processed/idx_user_map.json") as f:
    idx_user_map = json.load(f)
with open("../data/processed/idx_item_map.json") as f:
    idx_item_map = json.load(f)

print("Train matrix shape:", train_matrix.shape)
print("Train matrix nnz:", train_matrix.nnz)
print("Users mapped:", len(user_idx_map))
print("Items mapped:", len(item_idx_map))

Train matrix shape: (50000, 39865)
Train matrix nnz: 1148447
Users mapped: 50000
Items mapped: 39865


In [2]:
item_popularity = np.asarray(train_matrix.sum(axis=0)).flatten()

print("Popularity vector shape:", item_popularity.shape)
print("Min interactions:", item_popularity.min())
print("Max interactions:", item_popularity.max())
print("Mean interactions per item:", item_popularity.mean().round(2))

Popularity vector shape: (39865,)
Min interactions: 1.0
Max interactions: 4747.0
Mean interactions per item: 28.81


In [4]:
popularity_ranking = np.argsort(item_popularity)[::-1]

print("Top 10 item indices by popularity:", popularity_ranking[:10])
print("Top 10 popularity scores:", item_popularity[popularity_ranking[:10]])

Top 10 item indices by popularity: [13997 31025 22178 24335 17473 16018 14799 30687 22535 13037]
Top 10 popularity scores: [4747. 4257. 3998. 3283. 3281. 3214. 3207. 3045. 2942. 2915.]


In [6]:
import pandas as pd

news = pd.read_parquet("../data/processed/news.parquet")

top10_item_ids = [idx_item_map[str(i)] for i in popularity_ranking[:10]]
top10_news = news[news["news_id"].isin(top10_item_ids)][["news_id", "category", "subcategory", "title"]]

display(top10_news)
print("Category counts (top 10):")
print(top10_news["category"].value_counts())

,news_id,category,subcategory,title
2077,N31801,news,newspolitics,Joe Biden reportedly denied Communion at a Sou...
6892,N55189,tv,tvnews,"'Wheel Of Fortune' Guest Delivers Hilarious, O..."
11465,N306,movies,movies-celebrity,Kevin Spacey Won't Be Charged in Sexual Assaul...
11870,N29177,tv,tv-celebrity,"Miguel Cervantes' Wife Reveals Daughter, 3, 'D..."
14216,N42620,lifestyle,lifestylebuzz,Heidi Klum's 2019 Halloween Costume Transforma...
16666,N43142,sports,basketball_nba,Former NBA first-round pick Jim Farmer arreste...
21466,N45794,news,newscrime,Four flight attendants were arrested in Miami'...
33899,N55689,sports,football_nfl,"Charles Rogers, former Michigan State football..."
41550,N33619,news,newsus,College gymnast dies following training accide...
50688,N35729,news,newsus,Porsche launches into second story of New Jers...


Category counts (top 10):
category
news         4
tv           2
sports       2
movies       1
lifestyle    1
Name: count, dtype: int64


In [7]:
np.save("../data/processed/popularity_scores.npy", item_popularity)
np.save("../data/processed/popularity_ranking.npy", popularity_ranking)

print("Saved popularity_scores.npy, shape:", item_popularity.shape)
print("Saved popularity_ranking.npy, shape:", popularity_ranking.shape)

Saved popularity_scores.npy, shape: (39865,)
Saved popularity_ranking.npy, shape: (39865,)


In [8]:
def recommend_popularity(user_id, k=10, exclude_seen=True):
    user_idx = user_idx_map.get(user_id)
    if user_idx is None:
        return []  # cold-start user, not in train

    if exclude_seen:
        seen_items = set(train_matrix[user_idx].indices)
    else:
        seen_items = set()

    recs = []
    for item_idx in popularity_ranking:
        if item_idx not in seen_items:
            recs.append(item_idx)
        if len(recs) == k:
            break
    return recs

In [9]:
sample_users = list(user_idx_map.keys())[:5]

for u in sample_users:
    rec_idx = recommend_popularity(u, k=5)
    rec_news_ids = [idx_item_map[str(i)] for i in rec_idx]
    rec_titles = news[news["news_id"].isin(rec_news_ids)]["title"].tolist()
    print(f"\nUser {u}:")
    for title in rec_titles:
        print(" -", title)


User U100:
 - Kevin Spacey Won't Be Charged in Sexual Assault Case After Accuser Dies
 - Heidi Klum's 2019 Halloween Costume Transformation Is Mind-Blowing   But, Like, What Is It?
 - Four flight attendants were arrested in Miami's airport after bringing in thousands in cash, police say
 - Charles Rogers, former Michigan State football, Detroit Lions star, dead at 38
 - Porsche launches into second story of New Jersey building, killing 2

User U1000:
 - Kevin Spacey Won't Be Charged in Sexual Assault Case After Accuser Dies
 - Heidi Klum's 2019 Halloween Costume Transformation Is Mind-Blowing   But, Like, What Is It?
 - Four flight attendants were arrested in Miami's airport after bringing in thousands in cash, police say
 - Charles Rogers, former Michigan State football, Detroit Lions star, dead at 38
 - Porsche launches into second story of New Jersey building, killing 2

User U10001:
 - Kevin Spacey Won't Be Charged in Sexual Assault Case After Accuser Dies
 - Heidi Klum's 2019 Hal

In [10]:
violations = 0
for u in sample_users:
    user_idx = user_idx_map[u]
    seen_items = set(train_matrix[user_idx].indices)
    rec_idx = recommend_popularity(u, k=10)
    overlap = seen_items.intersection(rec_idx)
    violations += len(overlap)

print("Seen-item leakage violations across sample users:", violations)
assert violations == 0

Seen-item leakage violations across sample users: 0


In [11]:
rows = []
for u in sample_users:
    rec_idx = recommend_popularity(u, k=10)
    rec_news_ids = [idx_item_map[str(i)] for i in rec_idx]
    for rank, nid in enumerate(rec_news_ids, start=1):
        rows.append({"user_id": u, "rank": rank, "news_id": nid})

sample_recs_df = pd.DataFrame(rows)
sample_recs_df.to_csv("../data/processed/popularity_sample_top10.csv", index=False)

display(sample_recs_df.head(20))

,user_id,rank,news_id
0,U100,1,N306
1,U100,2,N55689
2,U100,3,N42620
3,U100,4,N45794
4,U100,5,N35729
5,U100,6,N33619
6,U100,7,N31801
7,U100,8,N55189
8,U100,9,N43142
9,U100,10,N29177
